In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import monotonically_increasing_id

In [0]:
orders_df = spark.table("transportation.silver.orders_clean")

In [0]:
date_dim = orders_df \
    .select("order_date") \
    .dropDuplicates() \
    .withColumn("year", year("order_date")) \
    .withColumn("month", month("order_date")) \
    .withColumn("day", dayofmonth("order_date")) \
    .withColumn("date_key", monotonically_increasing_id())

In [0]:
date_dim.write.mode("overwrite").saveAsTable("transportation.gold.dim_date")

In [0]:
orders_df = orders_df.withColumn("quantity", lit(1))

In [0]:
fact_orders = orders_df \
    .join(date_dim, "order_date") \
    .withColumn("fact_id", monotonically_increasing_id()) \
    .select(
        "fact_id",
        "order_id",
        "customer_id",
        "product_id",
        "date_key",
        "amount",
        "quantity"
    )

In [0]:
fact_orders.write.mode("overwrite").saveAsTable("transportation.gold.fact_orders")

In [0]:
display(fact_orders)

fact_id,order_id,customer_id,product_id,date_key,amount,quantity
0,2,102,1002,0,300,1
1,1,101,1001,1,500,1
2,5,104,1001,2,900,1
3,4,101,1002,3,200,1
4,3,103,1003,4,700,1
